# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chapcoda/flyrank-ML-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Both questions below are asked the way I'd want mine asked — the paper is careful in
most places, which is exactly why these two stood out. Neither is a claim I think is
wrong; both are claims whose design doesn't yet carry the weight put on them.

### Finding A — ML Appendix, "What Predicts Health?"

**The finding.** A Random Forest predicting health score reports Average Position at
43% importance, Impressions at 32%, Scroll Depth at 15%, and CTR at 8%. The paper
describes the model as holdout-tested.

**Where does the label come from?** Health score is defined earlier in the same paper
as impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts).
The four features the model ranks highest are the four components the target is built
from, and their importance order tracks their construction weights closely. The model
isn't discovering which features predict health — it's recovering the formula that
defines it.

**My question.** The paper flags this itself, which I want to credit directly: it
states the target is partly constructed from these inputs and that importance is
descriptive rather than causal. My question is the follow-up it stops short of. If the
target is an arithmetic function of the features, what does the 80/20 holdout actually
validate? A holdout test answers "does this generalize to unseen rows," but here the
relationship is an identity that holds by definition on every row, seen or unseen. The
split confirms the model learned the formula, not that anything transfers.

**How I'd make it stronger.** Either drop position, impressions, CTR, and scroll depth
from the feature set and ask what predicts health *net of* its own components — that
version would be a real finding — or reframe the section as a decomposition of the
health score rather than a prediction task, and drop the holdout language, which
currently implies a validation the design can't provide.

**Why I'm confident this is a fair reading.** I hit the same structure in my own Week 5
model: my label is defined as May impressions relative to April, and April sits inside
my Feb–April feature window, so my features partly contain the label's own denominator.
I'm auditing that in Section 3 of this notebook. Same shape, and I didn't spot it in my
own work until I went looking.

### Finding B — Finding #4, "The Freshness Multiplier"

**The finding.** 365+ day content refreshed within 30 days shows a 3.2x health boost
(10.7 to 34.5) and 57x more impressions (71 to 4,039). The paper calls refresh timing
one of the strongest measured levers available, and it becomes Priority Action #1 in
the playbook.

**Does the validation design support the claim?** Two design issues sit between the
comparison and the conclusion, and neither is addressed where the finding is stated.

*Selection.* Refreshed pages weren't randomly assigned — an editor chose them. Editors
choose pages worth saving: ones with prior visibility, existing demand, or strategic
importance. The comparison group is old pages nobody chose. So some unknown share of
the 57x gap is the selection, not the refresh, and the two can't be separated in an
observational snapshot.

*Survivorship.* The paper's own Local Snapshot Rule states that extended cuts come
from an active-content subset filtered to impressions_90d > 0 and sessions_90d > 0.
Old pages that were refreshed and stayed dead have zero impressions, so they're
filtered out before the comparison runs. The refreshed group is conditioned on having
worked.

**My question.** How much of the 3.2x and 57x survives once the comparison is restricted
to pages that were comparable *before* the refresh? A matched comparison — refreshed
pages against unrefreshed pages with similar prior impressions, age, and word count —
would separate the refresh effect from the choosing. Without that, 57x is an upper
bound on an effect of unknown size, and it's carrying the paper's top recommendation.

**Why I think this criticism is worth making rather than nitpicking.** The paper
demonstrates it knows how to handle exactly this problem elsewhere. It catches the 283:1
ratio in the 361+ freshness bucket, explains it comes from a single declining page,
and demotes it out of headline status. It flags survivor bias by name in the age-freshness
matrix and tells the reader not to use that cell as a decay proof point. That discipline
is real and consistent — it just doesn't reach the largest number in the paper. I'd
rather see the refresh claim stated as directional with the selection caveat attached
than see it dropped, because the underlying pattern is probably real. It's the size of
the number that the design can't support.

In [1]:
# ============ SETUP (self-contained — safe to re-run any time) ============
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

from huggingface_hub import login
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

DATASET = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"{DATASET}/dim_content.parquet"

FEAT_MONTHS = ["2026-02", "2026-03", "2026-04"]
FEAT_PATHS = [f"{DATASET}/fact_content_daily_performance/month={m}/data_0.parquet" for m in FEAT_MONTHS]
FEAT_UNION_SQL = " UNION ALL ".join([f"SELECT * FROM read_parquet('{p}')" for p in FEAT_PATHS])

print("Setup complete. Working dir:", os.getcwd())

# ============ FEATURE TABLE — Feb 1 - Apr 30 2026, per Week 3 data contract ============
q_features = f"""
WITH feature_window AS (
    {FEAT_UNION_SQL}
),
perf AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS impressions_90d,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_clicks ELSE 0 END) AS clicks_90d,
        SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_avg_position * gsc_impressions ELSE 0 END)
            / NULLIF(SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END), 0)
            AS avg_position_weighted,
        SUM(CASE WHEN ga4_data_available = TRUE THEN ga4_sessions ELSE 0 END) AS sessions_90d,
        SUM(CASE WHEN ga4_data_available = TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_sessions_90d,
        SUM(CASE WHEN ga4_data_available = TRUE THEN ga4_total_engagement_sec ELSE 0 END) AS engagement_sec_90d,
        SUM(CASE WHEN ga4_data_available = TRUE THEN sessions_organic ELSE 0 END) AS sessions_organic_90d,
        BOOL_OR(gsc_data_available) AS any_gsc_available,
        BOOL_OR(ga4_data_available) AS any_ga4_available
    FROM feature_window
    GROUP BY content_hash_id, client_hash_id
),
dims AS (
    SELECT
        content_hash_id,
        client_hash_id,
        content_type,
        main_intent,
        backlinks,
        char_count,
        word_count,
        search_volume,
        competition,
        competition_level,
        cpc,
        DATE_DIFF('day', content_created_date, DATE '2026-04-30') AS content_age_days,
        DATE_DIFF('day', content_updated_date, DATE '2026-04-30') AS days_since_last_update,
        DATE_DIFF('day', last_optimized_date, DATE '2026-04-30') AS days_since_last_optimized
    FROM read_parquet('{DIM_CONTENT}')
    WHERE is_published = TRUE
      AND is_deleted = FALSE
)
SELECT
    d.content_hash_id,
    d.client_hash_id,
    d.content_type,
    d.main_intent,
    d.backlinks,
    d.char_count,
    d.word_count,
    d.search_volume,
    d.competition,
    d.competition_level,
    d.cpc,
    d.content_age_days,
    d.days_since_last_update,
    d.days_since_last_optimized,
    COALESCE(p.impressions_90d, 0) AS impressions_90d,
    COALESCE(p.clicks_90d, 0) AS clicks_90d,
    p.avg_position_weighted,
    COALESCE(p.sessions_90d, 0) AS sessions_90d,
    COALESCE(p.engaged_sessions_90d, 0) AS engaged_sessions_90d,
    COALESCE(p.engagement_sec_90d, 0) AS engagement_sec_90d,
    COALESCE(p.sessions_organic_90d, 0) AS sessions_organic_90d,
    COALESCE(p.any_gsc_available, FALSE) AS any_gsc_available,
    COALESCE(p.any_ga4_available, FALSE) AS any_ga4_available
FROM dims d
LEFT JOIN perf p ON d.content_hash_id = p.content_hash_id
WHERE d.days_since_last_update >= 0
"""

features = con.execute(q_features).fetchdf()

features["ctr"] = np.where(
    features["impressions_90d"] > 0,
    features["clicks_90d"] / features["impressions_90d"],
    0.0,
)
features["engagement_rate"] = np.where(
    features["sessions_90d"] > 0,
    features["engaged_sessions_90d"] / features["sessions_90d"],
    0.0,
)

print(f"Feature table: {len(features):,} rows, {features['client_hash_id'].nunique()} clients")

Setup complete. Working dir: /content/flyrank-ml-internship-starter


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature table: 50,840 rows, 37 clients


In [2]:
# ============ LABEL — is_declining_label, per Week 3 data contract ============
TRAILING_PATH = f"{DATASET}/fact_content_daily_performance/month=2026-04/data_0.parquet"
TARGET_PATH   = f"{DATASET}/fact_content_daily_performance/month=2026-05/data_0.parquet"

q_label = f"""
WITH trailing_30d AS (
    SELECT content_hash_id,
           SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS trailing_impressions
    FROM read_parquet('{TRAILING_PATH}')
    GROUP BY content_hash_id
),
target_window AS (
    SELECT content_hash_id,
           SUM(CASE WHEN gsc_data_available = TRUE THEN gsc_impressions ELSE 0 END) AS target_impressions,
           BOOL_OR(gsc_data_available) AS any_may_gsc
    FROM read_parquet('{TARGET_PATH}')
    GROUP BY content_hash_id
)
SELECT
    t.content_hash_id,
    t.trailing_impressions,
    COALESCE(g.target_impressions, 0) AS target_impressions,
    COALESCE(g.any_may_gsc, FALSE) AS any_may_gsc,
    CASE
        WHEN t.trailing_impressions > 0
            THEN CASE WHEN COALESCE(g.target_impressions, 0) <= t.trailing_impressions * 0.75
                      THEN 1 ELSE 0 END
        ELSE NULL
    END AS is_declining_label
FROM trailing_30d t
LEFT JOIN target_window g ON t.content_hash_id = g.content_hash_id
"""

labels = con.execute(q_label).fetchdf()

model_df = features.merge(
    labels[["content_hash_id", "is_declining_label", "any_may_gsc"]],
    on="content_hash_id", how="inner")
model_df = model_df.dropna(subset=["is_declining_label"]).copy()
model_df["is_declining_label"] = model_df["is_declining_label"].astype(int)

print(f"Final modeling table: {len(model_df):,} rows, {model_df['client_hash_id'].nunique()} clients")
print(f"Declining rate: {model_df['is_declining_label'].mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final modeling table: 33,296 rows, 29 clients
Declining rate: 50.2%


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
# ============ SECTION 2 — NAIVE SPLIT VS HONEST SPLIT ============
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
import numpy as np, pandas as pd

# days_since_last_optimized deliberately excluded — 100% null, verified inert (see Section 3)
numeric_features = [
    "char_count", "word_count", "search_volume", "competition", "cpc",
    "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "avg_position_weighted",  "sessions_90d",
    "engaged_sessions_90d", "engagement_sec_90d", "sessions_organic_90d",
]

categorical_features = ["content_type", "main_intent", "competition_level"]
ALL_FEATURES = numeric_features + categorical_features

def build_model():
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ])
    return Pipeline([("preprocess", pre), ("clf", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                random_state=42, n_jobs=1))])

def precision_at_k(df, score_col, k=50):
    return df.sort_values(score_col, ascending=False).head(k)["is_declining_label"].mean()

def evaluate(train_df, test_df, label):
    m = build_model()
    m.fit(train_df[ALL_FEATURES], train_df["is_declining_label"])
    scored = test_df.copy()
    scored["model_score"] = m.predict_proba(scored[ALL_FEATURES])[:, 1]
    return {
        "split": label,
        "test_rows": len(scored),
        "test_clients": scored["client_hash_id"].nunique(),
        "base_rate": round(scored["is_declining_label"].mean(), 3),
        "precision_at_50": round(precision_at_k(scored, "model_score"), 3),
        "roc_auc": round(roc_auc_score(scored["is_declining_label"], scored["model_score"]), 3),
    }, scored

# --- BEFORE: naive random row-level split (pages from the same client land in both sides) ---
naive_train, naive_test = train_test_split(
    model_df, test_size=0.2, random_state=42, stratify=model_df["is_declining_label"])
naive_result, _ = evaluate(naive_train, naive_test, "naive random (row-level)")

# --- AFTER: client-grouped split (no client appears in both train and test) ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(splitter.split(model_df, groups=model_df["client_hash_id"]))
grouped_train = model_df.iloc[tr_idx].reset_index(drop=True)
grouped_test  = model_df.iloc[te_idx].reset_index(drop=True)
grouped_result, grouped_scored = evaluate(grouped_train, grouped_test, "client-grouped (honest)")

before_after = pd.DataFrame([naive_result, grouped_result])
print("BEFORE / AFTER — same model, same features, same metric, different split:\n")
print(before_after.to_string(index=False))

# --- Is the honest split's Precision@50 concentrated in one client? ---
top50 = grouped_scored.sort_values("model_score", ascending=False).head(50)
print(f"\nHonest split test set: {grouped_scored['client_hash_id'].nunique()} clients, "
      f"{len(grouped_scored):,} rows")
print("Top 50 predictions by client:")
print(top50["client_hash_id"].value_counts().to_string())
print(f"\nLargest single client's share of top 50: "
      f"{top50['client_hash_id'].value_counts().iloc[0] / 50:.0%}")

BEFORE / AFTER — same model, same features, same metric, different split:

                   split  test_rows  test_clients  base_rate  precision_at_50  roc_auc
naive random (row-level)       6660            28      0.502             0.92    0.729
 client-grouped (honest)       2477             6      0.506             0.60    0.632

Honest split test set: 6 clients, 2,477 rows
Top 50 predictions by client:
client_hash_id
client_3197e6291363b4db    47
client_8ae2bfb5aa1ffa1e     3

Largest single client's share of top 50: 94%


### The improvement: naive random split vs client-grouped split

| split | test rows | test clients | base rate | Precision@50 | ROC AUC |
|---|---|---|---|---|---|
| naive random (row-level) | 6,660 | 28 | 0.502 | **0.92** | 0.729 |
| client-grouped (honest) | 2,477 | 6 | 0.506 | **0.60** | 0.632 |

Same model, same features, same metric, same seed. The only thing that changed is
whether pages belonging to the same client can appear on both sides of the split.

Under the naive row-level split, Precision@50 measures 0.92. Under the honest
client-grouped split it measures 0.60 — the naive number is inflated by roughly 53%.
The mechanism is that pages from one client share a great deal: the same site
architecture, the same publishing cadence, the same GSC connection quality, often the
same seasonal demand shape. When a random split puts some of a client's pages in
training and the rest in test, the model can recognise the client rather than the
pattern. Nothing about that transfers to a client it has never seen, which is the only
situation that matters in deployment.

AUC tells the same story more modestly: 0.729 naive versus 0.632 honest. The inflation
is real in both metrics, but Precision@50 exaggerates it — a 15% relative gap on AUC
against 53% on Precision@50. That is the first sign of something the third finding
below develops further: the top-50 cutoff amplifies whatever is happening in the
ranking, in both directions.

I want to be precise about what this does and does not show. It does not show the model
is broken. It shows that the number I would have reported under a row-level split was
measuring something other than what I would have claimed it measured.

### What the honest number is actually measuring

The concentration check is the part of this section I did not expect. Of the model's
top 50 predictions on the honest test set, **47 come from a single client**
(`client_3197e6291363b4db`); three come from a second client, and the remaining four
test clients contribute none.

So 0.60 is very close to a measurement of one client's pages. It is not evidence that
the model ranks well across clients, because five of the six test clients are barely
represented at the top of the ranking at all.

This traces directly back to the split design. A 20% client holdout on 29 clients
leaves 6 clients in test, and those 6 differ enormously in how many pages they have and
how much GSC-honest coverage they carry. Whichever client contributes the most
high-impression pages will dominate a top-50 cut. Precision@50 has no mechanism to
notice this — it takes the global top 50 regardless of who owns those rows.

The honest read: **the client-grouped Precision@50 of 0.60 is directional evidence at
best, and it is measuring one client.** A more defensible design would compute
Precision@K per client and report the distribution, or use grouped cross-validation
across several folds so no single client's composition drives the headline. I have not
done that here; I am recording it as the correct next step rather than claiming the
current number is something it isn't.

### A third finding: the metric itself is fragile

Across four runs of what should be an identical pipeline — same seed, same split, same
rows, verified identical (33,296 rows, 29 clients, 50.19% declining, train 30,819 /
test 2,477, same six test client IDs) — Precision@50 has measured 0.520, 0.660, 0.580,
and 0.600. Within a single session it is perfectly reproducible: three consecutive fits
returned 0.58 and 0.633 identically. The variation is across sessions and settings, not
across repeated fits.

ROC AUC over the same four runs: 0.634, 0.634, 0.633, 0.634. Essentially unchanged.

That contrast is the finding. AUC uses the entire ranking, so small perturbations in
predicted probabilities move it hardly at all. Precision@50 is a hard cutoff at rank
50, so a handful of pages crossing that boundary swings it by several points.

The cleanest demonstration came last. Holding everything constant within a single
session and changing only `n_jobs` from -1 to 1 — a parallelism setting that should have
no mathematical effect whatsoever — moved Precision@50 by two points and changed the
composition of the top 50. ROC AUC over the same change moved by 0.001. The likely
mechanism is floating-point accumulation order varying with thread count, which
perturbs borderline probabilities just enough to reshuffle the rank-50 boundary. I have
set `n_jobs=1` and recorded the sklearn version in the setup cell so this notebook's
numbers are reproducible.

The fragility is not confined to the honest split either. Across runs, the naive
split's Precision@50 has measured 0.86, 0.88, and 0.92 on identical data, while its
ROC AUC stayed between 0.729 and 0.733. Both splits show the same pattern: the rank-50
cutoff moves, the full-ranking metric does not.

### What I am taking forward

The three findings compound rather than sitting separately. The metric my lane is
scored on is computed at a cutoff sensitive enough to move several points on
environment settings that should not affect it, over a top-50 that is 94% one client,
on a test set of six clients. Each of those is defensible in isolation; together they
mean **Precision@50 on this split is a directional signal, not a measurement I would
put weight on.** ROC AUC of ~0.632 is the more stable summary and the one I will lead
with from here.

---

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# ============ SECTION 3 — LEAKAGE AND FEATURE AUDIT ============
import pandas as pd, numpy as np

print("=" * 70)
print("CHECK 1 — Label-derived features in the feature set")
print("=" * 70)
banned = ["trend_direction", "trend_pct", "is_declining_label",
          "target_impressions", "trailing_impressions", "any_may_gsc"]
present = [c for c in ALL_FEATURES if c in banned]
print(f"Features used: {len(ALL_FEATURES)}")
print(f"Label-derived features present: {present if present else 'none'}")

print("\n" + "=" * 70)
print("CHECK 2 — Null coverage across the feature set")
print("=" * 70)
null_share = model_df[ALL_FEATURES].isna().mean().sort_values(ascending=False)
print((null_share[null_share > 0] * 100).round(1).to_string() or "no nulls")
print(f"\n100% null: {[c for c in ALL_FEATURES if model_df[c].isna().all()]}")

print("\n" + "=" * 70)
print("CHECK 3 — Dishonest-zero trap: pages labelled declining with no honest May GSC")
print("=" * 70)
declining = model_df[model_df["is_declining_label"] == 1]
no_may = declining[~declining["any_may_gsc"]]
print(f"Pages labelled declining:            {len(declining):,}")
print(f"  ...with no honest May GSC data:    {len(no_may):,} ({len(no_may)/len(declining):.1%})")
print(f"Pages labelled stable:               {(model_df['is_declining_label']==0).sum():,}")
stable_no_may = model_df[(model_df['is_declining_label']==0) & (~model_df['any_may_gsc'])]
print(f"  ...with no honest May GSC data:    {len(stable_no_may):,}")

# Same check inside the honest test set's top 50
top50 = grouped_scored.sort_values("model_score", ascending=False).head(50)
print(f"\nWithin the model's top 50 (honest split):")
print(f"  labelled declining:                {int(top50['is_declining_label'].sum())}")
print(f"  ...of those, no honest May GSC:    {int(((top50['is_declining_label']==1) & (~top50['any_may_gsc'])).sum())}")

print("\n" + "=" * 70)
print("CHECK 4 — Does April sit inside both the features and the label?")
print("=" * 70)
print("Feature window: 2026-02-01 to 2026-04-30")
print("Label:          May impressions <= 0.75 x April trailing impressions")
print("April is therefore in BOTH the feature aggregates and the label denominator.\n")
for feat in ["impressions_90d", "clicks_90d", "avg_position_weighted"]:
    sub = model_df[[feat, "is_declining_label"]].dropna()
    r = np.corrcoef(sub[feat], sub["is_declining_label"])[0, 1]
    print(f"  corr({feat:24s}, label) = {r:+.4f}")

print("\nMean feature values by label:")
print(model_df.groupby("is_declining_label")[
    ["impressions_90d", "clicks_90d", "avg_position_weighted"]].mean().round(2).to_string())

CHECK 1 — Label-derived features in the feature set
Features used: 17
Label-derived features present: none

CHECK 2 — Null coverage across the feature set
char_count           52.2
word_count           52.2
competition_level     6.0
main_intent           6.0
search_volume         5.7
competition           5.7
cpc                   5.7

100% null: []

CHECK 3 — Dishonest-zero trap: pages labelled declining with no honest May GSC
Pages labelled declining:            16,711
  ...with no honest May GSC data:    2,964 (17.7%)
Pages labelled stable:               16,585
  ...with no honest May GSC data:    0

Within the model's top 50 (honest split):
  labelled declining:                30
  ...of those, no honest May GSC:    13

CHECK 4 — Does April sit inside both the features and the label?
Feature window: 2026-02-01 to 2026-04-30
Label:          May impressions <= 0.75 x April trailing impressions
April is therefore in BOTH the feature aggregates and the label denominator.

  corr(impres

Four checks on the final feature set and label. One passed cleanly, one found a
contamination that changes how I read my headline metric, one found a data-quality
problem I had not flagged before, and one tested a hypothesis I had going in and
rejected it.

### Check 1 — No label-derived features made it into the model. PASS.

The Week 3 data contract identified `trend_direction`, `trend_pct`, and
`is_declining_label` as label-derived and therefore banned from the feature set. All 17
features used were checked against that list plus the intermediate label columns
(`target_impressions`, `trailing_impressions`, `any_may_gsc`). None are present. The
feature set draws only from the Feb–April window and static content metadata.

### Check 2 — Two features are more than half missing.

| feature | % null |
|---|---|
| char_count | 52.2 |
| word_count | 52.2 |
| competition_level | 6.0 |
| main_intent | 6.0 |
| search_volume | 5.7 |
| competition | 5.7 |
| cpc | 5.7 |

`char_count` and `word_count` are null for **52.2% of the modeling table**. The pipeline
imputes them with the column median, so the model trains on a value that is fabricated
for more than half the rows. This is not leakage, but it is a claim-quality problem: any
statement about content depth mattering in this model would be resting on imputed
values in the majority of cases. Neither feature appeared anywhere near the top of the
Week 5 permutation-importance ranking, which is consistent with them carrying little
real signal — but the honest reason is that half of what they carry is a constant.

Separately, `days_since_last_optimized` was 100% null in the Week 5 feature set. I
removed it and re-ran: Precision@50 and ROC AUC were identical to four decimal places.
The `SimpleImputer` had been silently skipping the column all along, so it never
influenced the model — it only produced two warnings per fit. It is excluded here, and
I now have evidence rather than an assumption for that decision.

### Check 3 — The label is contaminated by missing data, in one direction only.

This is the most consequential finding in the notebook.

The label is defined as May impressions ≤ 0.75 × April trailing impressions, and the
query uses `COALESCE(target_impressions, 0)`. So a page absent from the May partition
is treated as having zero May impressions. Zero always clears the 0.75× threshold.
**Missing May data can only ever produce a "declining" label; it can never produce a
"stable" one.**

The asymmetry is complete:

| | pages | with no honest May GSC data |
|---|---|---|
| labelled declining | 16,711 | 2,964 (17.7%) |
| labelled stable | 16,585 | 0 (0.0%) |

Week 3 established that `gsc_data_available = FALSE` produces dishonest zeros, and I
handled that correctly when building the *features* — every aggregate is guarded by a
`CASE WHEN gsc_data_available = TRUE` filter. I did not apply the same guard to the
*label*. The trap I identified in Week 3 caught me one notebook later, on the other
side of the pipeline.

Why it matters for the headline number: within the honest split's top 50, 30 pages are
llabelled declining, and **13 of those 30 have no honest May GSC data**. So 43% of what
counts as a correct prediction in Precision@50 may be recording a client's data
connection dropping rather than a page's search performance falling. I cannot tell from
this data which of those 13 actually declined.

The fix is straightforward and belongs in the next iteration: restrict the label to
pages with `any_may_gsc = TRUE`, so pages with no honest May coverage are excluded as
undefined rather than assumed to be zero. That is the same treatment `trailing_impressions
= 0` already receives. I have not re-run the model under the corrected label here —
that is a modelling change rather than an audit, and I would rather report the
contamination accurately than quietly patch it and present a new number.

### Check 4 — Mean reversion: hypothesis tested, not supported.

April sits inside both the feature window (Feb 1 – Apr 30) and the label's denominator
(April trailing impressions). My concern going in was that this could make the model
learn mean reversion — pages with high recent impressions have more room to fall — rather
than genuine decline risk.

| feature | correlation with label |
|---|---|
| impressions_90d | −0.0129 |
| clicks_90d | −0.0703 |
| avg_position_weighted | −0.1313 |

Mean values by class:

| | impressions_90d | clicks_90d | avg_position_weighted |
|---|---|---|---|
| stable | 2,860.79 | 9.19 | 20.84 |
| declining | 2,632.51 | 4.51 | 16.16 |

All three correlations are negative and weak. Declining pages have slightly *fewer*
impressions and materially fewer clicks than stable ones — the opposite direction from
what mean reversion would produce. The hypothesis is not supported and I am recording
it as tested and rejected rather than dropping it silently.

One result in that table is genuinely strange and I want to flag it rather than
explain it away: declining pages have a **better** average position (16.2 vs 20.8). Pages
ranking higher should not be more likely to decline. The most plausible reading connects
back to Check 3 — pages pulled into the declining class by missing May data carry
whatever Feb–April position they had, including good ones, so the contaminated 17.7%
would drag well-ranked pages into the declining group regardless of their actual
trajectory. That would make this an artifact of the label rather than a finding about
search. I cannot confirm that without re-running under the corrected label, so I am
stating it as the leading explanation, not the answer.

## 4. Claim rewrite


In [5]:
# ============ SECTION 4 — EVIDENCE BEHIND THE REWRITE ============
import pandas as pd

# Precision@50 vs ROC AUC across every configuration run on the SAME
# split, seed, and rows (33,296 rows / 29 clients / 6 test clients).
stability = pd.DataFrame([
    {"run": "w05, n_jobs=-1, incl. null feature",  "precision_at_50": 0.520, "roc_auc": 0.634},
    {"run": "w05 re-run, n_jobs=-1",               "precision_at_50": 0.660, "roc_auc": 0.634},
    {"run": "w06, n_jobs=-1, null feature dropped","precision_at_50": 0.580, "roc_auc": 0.633},
    {"run": "w06, n_jobs=1 (final)",               "precision_at_50": 0.600, "roc_auc": 0.634},
])
print("Metric stability across identical data and split:\n")
print(stability.to_string(index=False))
print(f"\nPrecision@50 range: {stability['precision_at_50'].max() - stability['precision_at_50'].min():.3f}")
print(f"ROC AUC range:      {stability['roc_auc'].max() - stability['roc_auc'].min():.3f}")

# What bounds the headline number
top50 = grouped_scored.sort_values("model_score", ascending=False).head(50)
declining_top50 = top50[top50["is_declining_label"] == 1]
unverifiable = declining_top50[~declining_top50["any_may_gsc"]]

print("\n" + "=" * 60)
print("Bounds on the reported Precision@50")
print("=" * 60)
print(f"Top 50 labelled declining:              {len(declining_top50)}")
print(f"  ...verifiable (honest May GSC):       {len(declining_top50) - len(unverifiable)}")
print(f"  ...unverifiable (missing May GSC):    {len(unverifiable)}")
print(f"\nReported Precision@50:                  {len(declining_top50)/50:.2f}")
print(f"Lower bound if all unverifiable are false positives: "
      f"{(len(declining_top50) - len(unverifiable))/50:.2f}")
print(f"\nTop-50 concentration: {top50['client_hash_id'].value_counts().iloc[0]}/50 "
      f"({top50['client_hash_id'].value_counts().iloc[0]/50:.0%}) from one client")

Metric stability across identical data and split:

                                 run  precision_at_50  roc_auc
  w05, n_jobs=-1, incl. null feature             0.52    0.634
               w05 re-run, n_jobs=-1             0.66    0.634
w06, n_jobs=-1, null feature dropped             0.58    0.633
               w06, n_jobs=1 (final)             0.60    0.634

Precision@50 range: 0.140
ROC AUC range:      0.001

Bounds on the reported Precision@50
Top 50 labelled declining:              30
  ...verifiable (honest May GSC):       17
  ...unverifiable (missing May GSC):    13

Reported Precision@50:                  0.60
Lower bound if all unverifiable are false positives: 0.34

Top-50 concentration: 47/50 (94%) from one client


### The claim as I wrote it in Week 5

> **Bottom line:** the Random Forest clearly outperforms the baseline rule on
> Precision@50 (0.660 vs. 0.260) and does so by finding a different, apparently more
> relevant signal than the one the hand-written rule relied on. The gain is real where
> it counts — 33 of the top 50 correct against a 50.6% base rate — but the moderate AUC
> (0.634) says the model separates well at the top of the ranking and much less sharply
> across the rest of the distribution.

Everything in that paragraph is arithmetically true of the run that produced it. The
problem is not accuracy, it is that the sentence carries more weight than the design
supports, in three specific ways this notebook measured.

### What the audit changed

**1. "0.660" is not a stable quantity.** Across four configurations of an otherwise
identical pipeline — same seed, same 33,296 rows, same 29 clients, same six test client
IDs — Precision@50 measured 0.520, 0.660, 0.580, and 0.600. Changing only `n_jobs` from
-1 to 1, a parallelism setting with no mathematical effect, moved it two points. Quoting
a three-significant-figure number implies a precision the measurement does not have.
ROC AUC over the same four runs: 0.634, 0.633, 0.634, 0.634.

**2. "33 of the top 50" is 47 pages from one client.** The honest split leaves six test
clients, and 94% of the top 50 belongs to a single one. The claim reads as a statement
about the model's ranking behaviour; the evidence is a statement about its behaviour on
one client's pages.

**3. "correct" is doing unearned work.** Of the 30 top-50 pages labelled declining, 12
have no honest May GSC data and were labelled declining because `COALESCE(May, 0)` maps
missing coverage to zero impressions. For 43% of the apparent hits, I cannot distinguish
a page that declined from a client whose data connection dropped.

### The rewrite

> **Bottom line:** on a client-held-out split, the Random Forest ranked at-risk pages
> better than the Week 4 baseline rule — Precision@50 of 0.60 against the baseline's
> 0.26, with ROC AUC of 0.634 against 0.298. The direction of that difference is
> consistent across every configuration I ran, and the AUC gap is the part I would rely
> on, since it moved by less than 0.002 across four runs while Precision@50 moved by
> eight points. Three limits bound what this measures. The top-50 cut is 94% one
> client, so it is not evidence of ranking quality across clients. Roughly 43% of the
> correctly-flagged pages in that cut were labelled declining on the basis of missing
> May GSC data rather than observed decline, so 0.60 is an upper bound — if every
> unverifiable page were a false positive, the floor would be 0.34.
> And Precision@50 itself proved sensitive to environment settings that should not
> affect it. The defensible reading is directional: the model orders pages better than
> the staleness rule did, on this data, under this label. It is decision-support for
> prioritising a review queue, not a measurement of how many declining pages exist.

### Claim language, before and after

| original | rewritten | why |
|---|---|---|
| "clearly outperforms" | "ranked at-risk pages better than" | Directional, not absolute. The direction held everywhere; the magnitude didn't. |
| "0.660" | "0.60" and the range stated | Two significant figures matches the measurement's stability. |
| "the gain is real where it counts" | limits stated before the reader draws conclusions | The original front-loads the win and buries the caveat. |
| "33 of the top 50 correct" | "roughly 43% ... labelled declining on the basis of missing data" | "Correct" implied a verified outcome that 13 of 30 do not have. |
| (absent) | "upper bound" | The contamination is one-directional, so the true figure can only be lower. |
| (absent) | "decision-support for prioritising a review queue" | Names the decision the number can actually support. |

### What I would not soften

Two claims survive the audit and I am keeping them at full strength.

The baseline comparison holds. A ROC AUC of 0.298 is not a weak result, it is an
inverted one — the staleness rule ranks pages consistently wrong, and inverting it
would score around 0.70. That is a real finding about the Week 4 rule and it does not
depend on any of the three limits above, because it is a property of the baseline's
ordering rather than of the top-50 cut.

The feature story holds too. `avg_position_weighted` and `ctr` carry essentially all the
signal; `days_since_last_update` — the feature the entire Week 4 rule was built on —
scored −0.0004 on permutation importance. That comes from AUC-based permutation over
the full test set, not from a top-50 cut, so the concentration and contamination
findings do not reach it.

Overcorrecting is its own failure. The audit narrowed one claim substantially; it did
not touch these two, and saying so is part of reporting it honestly.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.